# 01 — Fama-French Factor Construction & Validation

This notebook reconstructs the **SMB** (size) and **HML** (value) factors
from JKP data using the standard Fama-French 2×3 sort methodology, then
compares them against the official Fama-French factors.

## Methodology

1. **NYSE breakpoints**: median market_equity for size; 30th/70th percentile
   of book-to-market for value — computed using NYSE stocks only.
2. **2×3 sort**: intersect size (S/B) × value (G/N/V) → six portfolios.
3. **Value-weighted returns**: each stock's forward excess return weighted
   by its current market cap.
4. **Factor construction**:
   - SMB = (SV + SN + SG)/3 − (BV + BN + BG)/3
   - HML = (SV + BV)/2 − (SG + BG)/2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from quant_trading.data import load_jkp_csv, winsorize_returns, START_DATE, END_DATE
from quant_trading.factors import build_ff_factors, fetch_ff5_factors, compare_factors
from quant_trading.plotting import plot_factor_comparison

## 1. Load Data

Load the JKP dataset from CSV. If you need to re-download from WRDS,
see `quant_trading.data.JKP_SQL_TEMPLATE` for the query.

In [ ]:
# Update this path to your local JKP CSV file
JKP_CSV_PATH = "../data/jkp_data.csv"

df_jkp = load_jkp_csv(JKP_CSV_PATH)
df_jkp = winsorize_returns(df_jkp, lower=0.05, upper=0.95)

print(f"Rows: {len(df_jkp):,}")
print(f"Date range: {df_jkp['month_date'].min().date()} to {df_jkp['month_date'].max().date()}")
df_jkp.head()

## 2. Reconstruct SMB & HML

In [ ]:
subset = df_jkp.dropna(subset=["market_equity", "be_me", "ret_exc_lead1m", "exch_main"])
jkp_factors = build_ff_factors(subset, char_name="be_me")

print("Reconstructed JKP Factors (first 5 months):")
jkp_factors.head()

## 3. Compare Against Official FF Factors

In [ ]:
ff5 = fetch_ff5_factors(START_DATE, END_DATE)
corr_matrix = compare_factors(jkp_factors, ff5)

print("Correlation Matrix: Reconstructed JKP vs Official FF")
display(corr_matrix.round(4))

print(f"\nSMB correlation: {corr_matrix.loc['JKP_SMB', 'SMB']:.4f}")
print(f"HML correlation: {corr_matrix.loc['JKP_HML', 'HML']:.4f}")

## 4. Visual Comparison

In [ ]:
# Merge for aligned plotting
jkp_renamed = jkp_factors.rename(columns={"SMB": "JKP_SMB", "HML": "JKP_HML"})
merged = jkp_renamed.join(ff5[["SMB", "HML"]], how="inner").dropna()

fig = plot_factor_comparison(merged["JKP_SMB"], merged["SMB"], "SMB")
plt.show()

fig = plot_factor_comparison(merged["JKP_HML"], merged["HML"], "HML")
plt.show()